### RAG Pipelines -Data Ingestion to Vector DB Pipeline


In [25]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path



In [26]:
### Read all the pdf inside
def process_all_pdfs(pdf_directory):
    """process all pdf file in dir"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add sources information
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
                
            all_documents.extend(documents)
            print(f" Loaded: {len(documents)} pages")
            
        except Exception as e:
            print(f" Error loading {e}")

    print(f"\n Total documents loaded: {len(all_documents)}")
    return all_documents
#process all pdf in the data
all_pdf_documents = process_all_pdfs("../data")
    

In [27]:
all_pdf_documents

In [28]:
### Text Splitter get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """split document into small chuns for better rag"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    #show example of a chunk
    if split_docs:
        print(f"\n example chunk:")
        print(f"context: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata})")

    return split_docs


In [29]:
chunks = split_documents(all_pdf_documents)
chunks

### Embedding and vector store db

In [30]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [31]:
class EmbeddingManager:
    """Handles document embedding using senntencetrasformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initalize the embedding manager

        args:
            model_name: huggingface model name for sentence embedding
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loaded embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding Dimension: {self.model.get_sentence_embedding_dimension}")

           
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        generate embeddings for a list of texts

        Args:
            texts: List of strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)

        """

        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"generated embedding with shape: {embeddings.shape}")
        return embeddings

### initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager
    
        

### VectorStore

In [32]:
class VectorStore:
    """Manages documents embedding in a chromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        initialize the vector store

        Args:
            collection_name: name of the collection in chromaDB
            persist_directory: directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """" Initialize the chromaDB client and collection"""
        try:
            #create persistent chromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF documents vector store"}
            )
            print(f"vector store initalized. collection: {self.collection_name}")
            print(f"existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of document objects (with metadata)
            embeddings: numpy array of embeddings corresponding to the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match")

        print(f"adding {len(documents)} documents to vector store....")

        #prepare data for insertion
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #generate unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #Document Content
            documents_text.append(doc.page_content)

            #embedding
            embeddings_list.append(embedding.tolist())

        # add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
            )
            print(f"Successfully Added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore






In [33]:
chunks

In [34]:
### convert the text to embedding
texts=[doc.page_content for doc in chunks]

### generate the embeddings

embeddings=embedding_manager.generate_embeddings(texts)

### store int he vector db
vectorstore.add_documents(chunks,embeddings)


### Retriever Pipeline From VectorStore


In [35]:
class RAGRetriever:
    """Handles query-bashed retrieval from vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initalize the retriever

        Args:
            vector_store: Vector Store containing document embeddings
            embedding_manager:manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager


    def retrieve(self, query: str, top_k: int =5, score_threshold: float = 0.0) -> List[dict[str, Any]]:
        """
        reteieve relevant documents for a query

        Args:
            query: The Search query
            top_k:number of top result to return
            score_threshold:minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        
        """       

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, score threshold: {score_threshold}")

        #Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            #process  results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas,distances)):
                    #convert distance to similarity score (chromadb uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id' : doc_id,
                            'content' : document,
                            'metadata': metadata,
                            'similarity_score' : similarity_score,
                            'distance' : distance,
                            'rank' : i +1

                        })
                print(f"retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No document found")

            return retrieved_docs

        except Exception as e:
            print(f"error during retrieval: {e}")
            return []


rag_retriever=RAGRetriever(vectorstore, embedding_manager)
                   

In [36]:

rag_retriever


In [37]:
print(vectorstore.collection.count())

In [38]:
rag_retriever.retrieve("explain unit 1.2 topic", top_k=5, score_threshold=-1.0)

In [39]:
rag_retriever.retrieve("python", top_k=3, score_threshold=-1.0)

In [40]:
rag_retriever.retrieve(" tkinter framework with grid(")

### Integration Vectordb Context pipeline With LLM output


In [1]:
### simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the groq llm
groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(
    api_key=groq_api_key,
    model="openai/gpt-oss-120b",
    temperature=0.1,
    max_tokens=1024
)
 


###2.simple RAG function: retrieve +generate response

def rag_simple(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    prompt = f"""Use the following context to answer the question concisely.
Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(prompt)
    return response.content


In [42]:
answer = rag_simple("what is Tkinter Framework", rag_retriever, llm)
print(answer)

### Enhanced RAG Pipeline Features

In [43]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra featues:
    - retunes ans,sources,confidence score, and optionally full context.

    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'ans': 'no relevant context found.', 'sources': [], 'confidence': 0.0, 'context': "" }

    #prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source' : doc['metadata'].get('source_file', doc['metadata'].get('source','unknown')),
        'page' : doc['metadata'].get('page','unknown'),
        'score' : doc['similarity_score'],
        'preview' : doc['content'][:120] + '...'
    }for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # generate answer
    prompt = f"""use the following context to answer the question concisly.\n{context}\n\nQuestion: {query}\n\nans"""
    response = llm.invoke([prompt.format(context=context,query=query)])

    output = {
        'ans' : response.content,
        'sources' : sources,
        'confidence' : confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Software Development Life-Cycle Models ", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['ans'])
print("sources:",result['sources'])
print("confidence", result['confidence'])
print("context preview:", result['context'][:300])




In [46]:
from typing import List, Dict, Any
import time


class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(
        self,
        question: str,
        top_k: int = 5,
        min_score: float = 0.2,
        stream: bool = False,
        summarize: bool = False,
    ):
        results = self.retriever.retrieve(question, top_k, score_threshold=min_score)

        if not results:
            answer = "No relevant context found."
            answer_with_citations = answer
            sources = []
            summary = None
        else:
            context = "\n\n".join([doc["content"] for doc in results])
            sources = [
                {
                    "source": doc["metadata"].get(
                        "source_file", doc["metadata"].get("source", "unknown")
                    ),
                    "page": doc["metadata"].get("page", "unknown"),
                    "score": doc["similarity_score"],
                    "preview": doc["content"][:120] + "...",
                }
                for doc in results
            ]

            prompt = (
                f"Use the following context to answer the question concisely.\n"
                f"Context:\n{context}\n\nQuestion: {question}"
            )

            if stream:
                print("Streaming prompt preview:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i : i + 80], end="", flush=True)
                    time.sleep(0.05)
                print()

            response = self.llm.invoke([prompt])
            answer = response.content

            citations = [
                f"[{i+1}] {src['source']} (page {src['page']})"
                for i, src in enumerate(sources)
            ]
            answer_with_citations = (
                answer + "\n\nCitations:\n" + "\n".join(citations)
                if citations
                else answer
            )

            summary = None
            if summarize and answer:
                summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
                summary_resp = self.llm.invoke([summary_prompt])
                summary = summary_resp.content

        self.history.append(
            {
                "question": question,
                "answer": answer,
                "sources": sources,
                "summary": summary,
            }
        )

        return {
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary,
            "history": self.history,
        }


adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query(
    "Software Development Life-Cycle Models",
    top_k=3,
    min_score=0.1,
    stream=True,
    summarize=True,
)
print("\nFinal Answer:\n", result["answer"])
print("\nSummary:\n", result["summary"])
print("\nLatest History:\n", result["history"][-1])